In [1]:
from pathlib import Path
import pandas as pd

data_path = Path("data/processed/tuen_mun_aqhi.csv")
if not data_path.exists():
    data_path = Path("tuen_mun_aqhi.csv")
if not data_path.exists():
    raise FileNotFoundError(
        "Upload tuen_mun_aqhi.csv in Colab, or run from the repository root."
    )

data = pd.read_csv(
    data_path,
    dtype={"aqhi": "float64", "is_substituted": "bool"},
)
data["timestamp"] = pd.to_datetime(
    data["timestamp"], utc=True, errors="raise"
).dt.tz_convert("Asia/Hong_Kong")
data = data.sort_values("timestamp").reset_index(drop=True)

print("First five observations:")
print(data.head().to_string(index=False))


First five observations:
                timestamp  aqhi  is_substituted
2025-04-01 01:00:00+08:00   2.0           False
2025-04-01 02:00:00+08:00   2.0           False
2025-04-01 03:00:00+08:00   2.0           False
2025-04-01 04:00:00+08:00   2.0           False
2025-04-01 05:00:00+08:00   2.0           False


In [2]:
assert list(data.columns) == ["timestamp", "aqhi", "is_substituted"]
assert not data.empty
assert data["timestamp"].notna().all()
assert data["timestamp"].is_unique, "Review duplicate timestamps."

expected_hours = pd.date_range(
    start=data["timestamp"].min(),
    end=data["timestamp"].max(),
    freq="h",
)
missing_hours = expected_hours.difference(pd.DatetimeIndex(data["timestamp"]))
assert len(missing_hours) == 0, "Review missing hours before building targets."

available_aqhi = data["aqhi"].dropna()
assert available_aqhi.isin(range(1, 12)).all(), "Review AQHI codes."

quality_summary = pd.Series({
    "Hourly rows": len(data),
    "Available AQHI readings": int(data["aqhi"].notna().sum()),
    "Missing AQHI readings": int(data["aqhi"].isna().sum()),
    "Duplicate timestamps": int(data["timestamp"].duplicated().sum()),
    "Missing hours": len(missing_hours),
    "Substituted readings": int(data["is_substituted"].sum()),
    "10+ category encoded as 11": int(data["aqhi"].eq(11).sum()),
})
print(quality_summary.to_string())
print("First timestamp:", data["timestamp"].min())
print("Last timestamp:", data["timestamp"].max())


Hourly rows                   10944
Available AQHI readings       10926
Missing AQHI readings            18
Duplicate timestamps              0
Missing hours                     0
Substituted readings           1083
10+ category encoded as 11       15
First timestamp: 2025-04-01 01:00:00+08:00
Last timestamp: 2026-07-01 00:00:00+08:00


In [3]:
high_count = int(available_aqhi.ge(7).sum())
other_count = int(available_aqhi.lt(7).sum())
class_counts = pd.Series({
    "AQHI below 7": other_count,
    "AQHI at least 7": high_count,
})
print(class_counts.to_string())
print(f"High-AQHI share among available readings: {high_count / len(available_aqhi):.2%}")
print(f"Available readings below 7: {other_count / len(available_aqhi):.2%}")
print("Missing measurements remain missing:", int(data["aqhi"].isna().sum()))


AQHI below 7       10621
AQHI at least 7      305
High-AQHI share among available readings: 2.79%
Available readings below 7: 97.21%
Missing measurements remain missing: 18
